# 方策反復法

方策反復法は「現在の方策を評価」してから「より良い方策へ改善」する2段階を繰り返します。改善手順が分かれているため、実装の責務分離が明確になります。


## 参考動画
- [対応動画 1](https://www.youtube.com/watch?v=iMqByZlwHvA)
@[youtube](https://www.youtube.com/watch?v=iMqByZlwHvA)

## 参考リンク
- https://www.youtube.com/watch?v=iMqByZlwHvA


## 背景と目的

評価と改善を分離すると、どこで性能が詰まっているかを切り分けやすくなります。

この性質は、業務環境でアルゴリズムを監査可能な形で運用する際に有利です。

2状態MDPで方策反復を最後まで実行し、収束方策を確認します。


## 最初に解きたい疑問

1. `eval_policy` を何回回せば、評価が十分と言えるのか。
2. `policy stable` は、何を見て止めているのか。
3. `P[s][a]` の2つの遷移は、何を表しているのか。
4. 評価と改善を分けると、何がそんなに扱いやすくなるのか。
5. 方策反復が最適解に近づく理由を、どう理解すればよいのか。


## 先に押さえる言葉

- `policy evaluation`: 今の方策がどれくらい良いかを、価値関数として見積もる手続きです。方策は変えずに価値だけを更新します。
- `policy improvement`: 評価した価値を使って、より良い行動に方策を更新することです。各状態での選択を見直します。
- `stable policy`: 改善しても方策が変わらなくなった状態です。これ以上の更新が止まる目安になります。
- `expected action value`: ある状態である行動を取ったときの、期待される将来価値です。`action_value` が計算している量です。


## 実行前の見取り図

1. `iter=... policy=... V=...` で、評価と改善が交互に進んでいるか。
2. `policy stable -> stop` が出るかどうか。
3. 最終的な policy と `V` の組が、収束結果として読めるか。


## つまずきやすい点

- `action_value` が期待値計算になっている理由を、確率付き遷移に結びつけて説明する必要がある。
- 2状態MDPで見える収束を、一般の状態数ではどう解釈するかの補足がない。


## コード 1: 更新式や補助関数を定義する

このセルでは 更新式や補助関数を定義する ための最小コードを動かします。 実行時は「`iter=... policy=... V=...` で、評価と改善が交互に進んでいるか。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
states = ['s0', 's1']
actions = ['left', 'right']
gamma = 0.9

# P[s][a] = (next_state, reward)
P = {
    's0': {
        'left': [('s0', 0.2), ('s1', 1.0)],  # deterministic via two branches in expectation example
        'right': [('s1', 0.4), ('s1', 0.4)],
    },
    's1': {
        'left': [('s0', 0.0), ('s0', 0.0)],
        'right': [('s1', 0.8), ('s1', 0.8)],
    },
}

def action_value(s, a, V):
    # two equally likely transitions for toy example
    trans = P[s][a]
    q = 0.0
    for ns, r in trans:
        q += 0.5 * (r + gamma * V[ns])
    return q

def eval_policy(policy, V, n_iter=40):
    for _ in range(n_iter):
        newV = {}
        for s in states:
            a = policy[s]
            newV[s] = action_value(s, a, V)
        V = newV
    return V

def improve_policy(V):
    policy = {}
    stable = True
    for s in states:
        q_left = action_value(s, 'left', V)
        q_right = action_value(s, 'right', V)
        best = 'left' if q_left >= q_right else 'right'
        policy[s] = best
    return policy


## コード 2: 方策評価と改善の準備をする

このセルでは 方策評価と改善の準備をする ための最小コードを動かします。 実行時は「`policy stable -> stop` が出るかどうか。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
V = {'s0': 0.0, 's1': 0.0}
policy = {'s0': 'left', 's1': 'left'}

for k in range(8):
    V = eval_policy(policy, V)
    new_policy = improve_policy(V)
    print(f'iter={k}: policy={policy}, V={{s0:{V["s0"]:.4f}, s1:{V["s1"]:.4f}}}')
    if new_policy == policy:
        print('policy stable -> stop')
        break
    policy = new_policy
